실습 5. 누락 시점 보간 후 재분석
- 채우는 방법에 따라 결과가 어떻게 다른지 비교

목표
- 누락을 만들고 채우는 방법에 따라 결과가 어떻게 다른지 비교

단계
- 작은 예시에서 앞값 채움과 선형 보간의 결과 차이를 비교
- 시각 데이터 일부를 일부러 비우고 누락 개수를 확인
- 보간으로 채운 뒤 누락이 사라졌는지 확인

예상 결과
- 앞값 채움은 계단 모양, 선형 보간은 12·14로 이어짐; 보간 후 0

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("../data/21_engine1_timestamp_sample.csv")

In [2]:
# [결측치 보정 기법 — 직전값 대체(ffill) vs 선형보간(interpolate)]
# 1. ffill() (Forward Fill): 가장 최근에 정상 측정된 앞의 값을 그대로 끌어다 결측 구간에 덮어씁니다.
# 2. interpolate() (Linear Interpolation): 결측 구간의 이전 정상값과 이후 정상값 사이를 일정한 경사로 이어 채웁니다.
# 3. s2.iloc[50:55] = np.nan: 의도적으로 5개 행에 결측을 주입한 후 interpolate를 통해 복원 테스트를 진행합니다.
# * 통신 장비 오류나 센서 재부팅으로 인해 누락된 구간이 존재하면 분석 함수(특히 시계열 분해나 머신러닝)가
#   작동하지 않고 에러를 내므로, 시간적 흐름에 부합하도록 주변 수치를 추정하여 견고하게 결측을 제거해야 합니다.
# * ffill()과 interpolate() 모두 복원 결과를 기존 시리즈나 데이터프레임에 재할당해주거나
#   `inplace=True` 파라미터를 넘겨주어야 원본이 수정됩니다.
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')

s = pd.Series([10.0, np.nan, np.nan, 16.0, 18.0])
print('ffill:', s.ffill().tolist()) # ffill: [10.0, 10.0, 10.0, 16.0, 18.0]
print('interpolate:', s.interpolate().tolist()) # interpolate: [10.0, 12.0, 14.0, 16.0, 18.0]

s2 = df['s_4'].copy(); s2.iloc[50:55] = np.nan
print('누락 개수:', int(s2.isna().sum())) # 누락 개수: 5
print('보간 후:', int(s2.interpolate().isna().sum())) # 보간 후: 0

ffill: [10.0, 10.0, 10.0, 16.0, 18.0]
interpolate: [10.0, 12.0, 14.0, 16.0, 18.0]
누락 개수: 5
보간 후: 0
